In [16]:
import numpy as np
import torch as th
from PIL import Image
import json, glob, os, tqdm

sj_path = "/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/sample_json/TPAMI_MajorRevision/multipie_validset2.json"

method = ["DPP_CTRL_128_VLL_ch128_attn16-8__SD256_250T_dstC", "CTRL_With_DPPNonSpa_attn16-8_50k", "DPP_CTRL_no_hint_block_128_VLL_ch128_attn16-8__SD256_250T_dstC_50k", "DPP_Spatial_with_CA_attn16-8__SD256_250T_dstC", "DPP_128_50k"]
sample = json.load(open(sj_path, "r"))
meta = json.load(open("./mp_controlnet_targetSH.json", "r"))
os.makedirs("./mp_targetSH_figures/", exist_ok=True)

source = "/data/mint/DPM_Dataset/MultiPIE/MultiPIE_validset2/mp_aligned_128/valid/"

counter = 0
for pid, dat in tqdm.tqdm(sample['pair'].items()):
    src = dat['src']
    dst = dat['dst']
    
    input_img = np.array(Image.open(f'{source}/{src}'))
    gt_img = np.array(Image.open(f'{source}/{dst}'))

    out = [input_img, gt_img]
    for m in method:
        meta_dat = meta[m]
        img_dir = meta_dat['img_dir']
        n_frame = int(meta_dat.get('n_frame', 2))

        fn = f"input={src}#pred={dst}"
        relit_img = f'{img_dir}/{fn}.png'
        
        if not os.path.exists(relit_img):
            relit_img = Image.fromarray(np.zeros((128, 128, 3), dtype=np.uint8))
        else:
            relit_img = Image.open(relit_img)
            
        out.append(relit_img)

    out = np.concatenate(out, axis=1)

    Image.fromarray(out).save(f"./mp_targetSH_figures/{pid}_res.png")

100%|██████████| 100/100 [00:03<00:00, 25.89it/s]


In [ ]:
import matplotlib.pyplot as plt
grid = [
        [2302, 2861, 286, '_unknown', 2913, 1587, 1640],
        [1264, 553, 2554, 1901, 1494, 2015, 1016],
        [2059, 1186, 1355, 1581, 1812, 1838, 2076],
        [1060, 1566, 2245, 2285, 2317, 2412, 2512]
    ]
img_path = '/data/mint/DPM_Dataset/ffhq_256_with_anno/ffhq_256/valid/'
os.makedirs("./targetSH_figures_final_grid/", exist_ok=True)

# print(sample['pair'].keys())
for i, each_grid in enumerate(grid):
    grid = []
    for e in each_grid:
        src = sample['pair'][f'pair{e}']['src']
        dst = sample['pair'][f'pair{e}']['dst']
        src_img = Image.open(f"{img_path}/{src}")
        dst_img = Image.open(f"{img_path}/{dst}")
        r = 0.45
        offset = 10
        dst_img = dst_img.resize((int(dst_img.size[0]*r), int(dst_img.size[1]*r)))
        # Put src_img and dst_img together in white canvas 256 x (256 + dst_img.width)
        canvas = Image.new('RGB', (256 + dst_img.size[0] + offset, 256), (255, 255, 255))
        canvas.paste(src_img, (0, 0))
        canvas.paste(dst_img, (256, 256 - dst_img.size[1]))
        
        img = Image.open(f"./targetSH_figures/pair{e}_res.png")
        cond = Image.open(f"./targetSH_figures/pair{e}_cond.png")
        cond = cond.resize((cond.size[0]//2, cond.size[1]//2))
        # print(np.array(cond).shape)
        img = np.concatenate([np.array(canvas), np.array(img), np.array(cond)], axis=1)
        grid.append(img)
    grid = np.concatenate(grid, axis=0)
    Image.fromarray(grid).save(f"./targetSH_figures_final_grid/grid_row{i}.png")

In [ ]:
meta = json.load(open("./mp_controlnet_targetSH.json", "r"))
for m in meta:
    img_dir = meta[m]['img_dir']
    os.system(f"rsync -auz --progress mint@10.204.100.114:{img_dir} ./controlnet_mp/")